# Exercise 9 · Scan the code and decide each finding

Bandit, a security scanner for Python, reads `lookup.py` and reports four findings. You decide what each one means.

**Goal:** Scan a file with Bandit. Fix each finding, or keep it with a written reason.

**Why it matters:** A scanner only claims. A person reads the line and decides.

**Before you start:** You installed Codex and signed in (Monday's exercises).

**What the check measures:** `bandit.json` is newer than `lookup.py` and lists no finding. You mark each finding you keep in `lookup.py` with its id and a reason (Step 5). Your answer to one question is right.

**Done when:** the last cell prints `Done:`.

## The situation

`lookup.py` finds cases by status in a database and packs a folder with `tar`. A teammate wrote it fast. Before it merges, you scan it. The files hold made-up data.

## Step 1 · Write the practice files

This is the solved copy. Run the cell. It writes `practice/` as it is after Step 6: the fixed `lookup.py`, `bandit.json` and two commits. It never overwrites a file.

Expect three `Wrote practice/` lines and `Committed the start and the fixes`.

In [ ]:
FILES = {
    "lookup.py": """# Look up cases by status, and archive a folder. Made up for the exercise.
import os
import sqlite3
import subprocess  # nosec B404  # Kept: archive() needs it, and Bandit checks each call.

DB_PASSWORD = os.environ.get("DB_PASSWORD")


def find_cases(conn: sqlite3.Connection, status: str) -> list[tuple]:
    \"\"\"Find the cases that have one status.

    Args:
        conn: The open cases database.
        status: The status to match.

    Returns:
        The (id, status) rows that match.

    Raises:
        sqlite3.Error: If the query fails.
    \"\"\"
    query = "SELECT id, status FROM cases WHERE status = ?"
    return conn.execute(query, (status,)).fetchall()


def archive(folder: str) -> None:
    \"\"\"Pack a folder into archive.tgz in the current folder.

    Args:
        folder: The folder to pack.

    Raises:
        subprocess.CalledProcessError: If tar fails.
    \"\"\"
    subprocess.run(["tar", "czf", "archive.tgz", folder], check=True)  # nosec B603, B607  # Kept: no shell, and we trust this PATH.
""",
    ".gitignore": """__pycache__/
""",
    "bandit.json": """{
  "errors": [],
  "generated_at": "2026-11-06T03:30:00Z",
  "metrics": {
    "./lookup.py": {"CONFIDENCE.HIGH": 0, "CONFIDENCE.LOW": 0, "CONFIDENCE.MEDIUM": 0, "CONFIDENCE.UNDEFINED": 0, "SEVERITY.HIGH": 0, "SEVERITY.LOW": 0, "SEVERITY.MEDIUM": 0, "SEVERITY.UNDEFINED": 0, "loc": 26, "nosec": 0, "skipped_tests": 3},
    "_totals": {"CONFIDENCE.HIGH": 0, "CONFIDENCE.LOW": 0, "CONFIDENCE.MEDIUM": 0, "CONFIDENCE.UNDEFINED": 0, "SEVERITY.HIGH": 0, "SEVERITY.LOW": 0, "SEVERITY.MEDIUM": 0, "SEVERITY.UNDEFINED": 0, "loc": 26, "nosec": 0, "skipped_tests": 3}
  },
  "results": []
}
""",
}

from pathlib import Path

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

# Codex's fixes (Step 4), the kept findings (Step 5) and the saved scan (Step 6), simulated.
import os
import subprocess

START_PY = """# Look up cases by status, and archive a folder. Made up for the exercise.
import sqlite3
import subprocess

DB_PASSWORD = "hunter2"


def find_cases(conn: sqlite3.Connection, status: str) -> list[tuple]:
    \"\"\"Find the cases that have one status.

    Args:
        conn: The open cases database.
        status: The status to match.

    Returns:
        The (id, status) rows that match.

    Raises:
        sqlite3.Error: If the query fails.
    \"\"\"
    query = "SELECT id, status FROM cases WHERE status = '" + status + "'"
    return conn.execute(query).fetchall()


def archive(folder: str) -> None:
    \"\"\"Pack a folder into archive.tgz in the current folder.

    Args:
        folder: The folder to pack.

    Raises:
        subprocess.CalledProcessError: If tar fails.
    \"\"\"
    subprocess.run("tar czf archive.tgz " + folder, shell=True, check=True)
"""
if not (PRACTICE / ".git").exists():
    (PRACTICE / "lookup.py").write_text(START_PY, encoding="utf-8")
    for step in (["init", "-q", "-b", "main"], ["add", "lookup.py", ".gitignore"], ["commit", "-qm", "Start"]):
        subprocess.run(["git", "-C", str(PRACTICE), *step], check=True)
    (PRACTICE / "lookup.py").write_text(FILES["lookup.py"], encoding="utf-8")
    later = (PRACTICE / "lookup.py").stat().st_mtime + 1
    os.utime(PRACTICE / "bandit.json", (later, later))
    subprocess.run(["git", "-C", str(PRACTICE), "add", "-A"], check=True)
    subprocess.run(["git", "-C", str(PRACTICE), "commit", "-qm", "Fix or keep each Bandit finding"], check=True)
    print("Committed the start and the fixes")

## Step 2 · Commit the start, then scan

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-09-scan-and-decide/practice
git init -q -b main && git add -A && git commit -qm "Start"
uv run --with bandit==1.9.4 bandit -r .
```

`uv run --with` adds Bandit for this command only. `-r .` scans every Python file here.

Expect four `>> Issue:` blocks. Each gives a test id (Bandit's name for one of its checks), such as `B602`, and a severity.

## Step 3 · Decide each finding

A query is the SQL command that `find_cases` sends to the database.

| Test id | Bandit's claim |
| --- | --- |
| B602 | a call with `shell=True` |
| B608 | a query built from strings |
| B105 | a password in the code |
| B404 | the code imports `subprocess` |

In the cell, write fix or keep, and why, for each id. Then open the answer. The check does not read it.

<details><summary>Show the answer</summary>

- **B602: fix.** Why: a shell reads the folder name, so a `;` in it can start a second command.
- **B608: fix.** Why: a `'` in the status changes the query. A `?` placeholder (a mark the database fills with the value) keeps the value apart. Bandit shows Confidence Low, but the fault is real.
- **B105: fix.** Why: anyone who reads the code reads the password.
- **B404: keep.** Why: the import runs nothing. Other tests, such as B602, check each call.

</details>

In [ ]:
DECISIONS = {"B602": "fix: a ; in the folder name starts a command", "B608": "fix: a ' in the status changes the query",
             "B105": "fix: the code shows the password", "B404": "keep: the import runs nothing"}
print("Saved your decisions.")

### Learn: a finding is a claim to check

[Bandit's docs](https://bandit.readthedocs.io/en/1.9.4/config.html) allow `# nosec` only after a person reviews the line and accepts the finding.

## Step 4 · Ask Codex to fix three findings

```sh
codex --no-daemon
```

If Codex stops with an error about `--no-daemon`, run `codex` without it. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Type this request:

```text
In lookup.py, fix three Bandit findings. B608: use a ? placeholder. B602: run tar with a list of arguments, no shell=True. B105: read DB_PASSWORD from an environment variable (a value set outside the code) with os.environ.get. Add no # nosec.
```

Type `/diff` and read the change. Press `q`. Then type `/exit`. Expect a diff in `lookup.py` only.

## Step 5 · Scan again, and keep the rest with a reason

```sh
uv run --with bandit==1.9.4 bandit -r .
```

Expect B404, and two new findings on the `subprocess.run` line: B603 and B607.

To keep a finding, end its line in `practice/lookup.py` with `# nosec`, the ids, a second `#` and your reason:

```python
import subprocess  # nosec B404  # Kept: archive() needs it.
```

Name the ids: a bare `# nosec` also hides later findings. Save the file and scan again. Expect `No issues identified.`

<details><summary>Show the answer</summary>

- **B603: keep.** Why: `tar` gets a list of arguments and no shell, so a `;` stays part of the folder name.
- **B607: keep or fix.** Why: the terminal looks for `tar` in the PATH folders, so another `tar` found first would run. The full path `/usr/bin/tar` removes B607.

</details>

## Step 6 · Save the report, then commit

```sh
uv run --with bandit==1.9.4 bandit -r . -f json -o bandit.json
git add -A && git commit -qm "Fix or keep each Bandit finding"
```

Expect `JSON output written to file: bandit.json`.

## Step 7 · Answer one question

Your Step 5 scan said `No issues identified.` What do you know now?

- **A.** `lookup.py` is safe.
- **B.** You know about the scan, not about the code.
- **C.** The `# nosec` comments can go.

Write your letter between the quotes, then run the cell.

<details><summary>Show the answer</summary>

B. Why: Bandit runs a fixed set of tests. On `DB = "hunter2"` it reports nothing, because the name does not look like a password.

</details>

In [ ]:
ANSWER = "B"
print(f"Saved your answer: {ANSWER}")

## Start over (only if you need it)

Set `START_OVER = True` and run the cell. It writes the Step 1 files again. Set it back to `False`, then scan again (Step 2). Expect `Nothing changed.` while it is `False`.

In [ ]:
# Start over: only if START_OVER is True, write the Step 1 files again
START_OVER = False

if not START_OVER:
    print("Nothing changed.")
elif "FILES" not in globals():
    print("Run the Step 1 cell first.")
else:
    for name, text in FILES.items():
        (PRACTICE / name).write_text(text, encoding="utf-8")
    print("Wrote the files again. Set START_OVER back to False.")

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do.

In [ ]:
# Check your work: read practice/bandit.json and practice/lookup.py
import json
import re
from pathlib import Path

PRACTICE = Path("practice")
NOT_ALLOWED = [  # (pattern, problem)
    (r"shell=True", "lookup.py still has shell=True (Step 4)."),
    (r"""f["']SELECT|SELECT.*["'] *[+%]|SELECT.*\.format\(""", "lookup.py builds its query from strings (Step 4)."),
    (r"hunter2", "lookup.py still holds the password (Step 4)."),
    (r"#\s*nosec\b(?! *B\d{3})|#\s*nosec *(B\d{3}[ ,]*)+(# *)?$|#\s*nosec[^#\n]*B(602|608|105)",
     "a # nosec in lookup.py lacks its ids or a reason, or keeps B602, B608 or B105 (Steps 4 and 5)."),
]


def first_problem():
    """Print ok for each check that passes; return the first problem, or ""."""
    lookup, saved = PRACTICE / "lookup.py", PRACTICE / "bandit.json"
    if not saved.is_file() or saved.stat().st_mtime <= lookup.stat().st_mtime:
        return "no practice/bandit.json newer than lookup.py. Run the JSON command (Step 6)."
    report = json.loads(saved.read_text(encoding="utf-8"))
    if not any(name.endswith("lookup.py") for name in report.get("metrics", {})):
        return "bandit.json did not scan lookup.py. Run the JSON command from practice/ (Step 6)."
    for left in report.get("results", []):
        return f"bandit.json still lists {left.get('test_id')}. Fix or keep it (Steps 4 and 5), then Step 6."
    print("ok: bandit.json is newer than lookup.py, scanned it, and lists no finding")
    text = lookup.read_text(encoding="utf-8")
    for pattern, problem in NOT_ALLOWED:
        if re.search(pattern, text, re.MULTILINE):
            return problem
    print("ok: lookup.py has no shell, string-built query or password, and each # nosec has ids and a reason")
    choice = str(globals().get("ANSWER", "")).strip().upper()
    if not choice:
        return "the question has no answer. Write a letter in the Step 7 cell and run it."
    if choice != "B":
        return f"answer {choice} is not right. Read Step 7's answer, then try again."
    print("ok: your answer is right")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Run the Step 1 cell, then this cell."
print(f"Not yet: {problem}" if problem else "Done: each finding is fixed, or kept with a reason.")

## What you learned

- A finding is a claim to check. Fix it, or keep it with its ids and a reason.
- A silent scan tells you about the scan, not the code.

**Reflect:** Which finding would you show a teammate before a merge?